# Multi-Sensor Robustness Analysis: PCA vs VAE

## Problem Statement

**Critical Question**: If we train a model (PCA or VAE) on data from one sensor, can we apply it to data from other sensors?

### Real-World Scenario
- You develop a PSD signature detection system on Sensor 1
- Tomorrow, you deploy 10 more sensors
- Each sensor has slightly different characteristics:
  - Baseline power offset
  - Noise floor
  - Gain/calibration
  - Temperature drift

### Key Questions
1. Can the same **fitted model** work across sensors?
2. Can the same **hyperparameters** work across sensors?
3. What are viable **multi-sensor strategies**?

## Approach

1. Generate 10+ diverse PSD signatures
2. Simulate multiple sensors with different characteristics
3. Test cross-sensor model application
4. Explore solutions: normalization, retraining, pooled training

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import seaborn as sns
from typing import Tuple, List, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

# PyTorch imports for VAE
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)

# Plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Extended Signature Library (10+ Signatures)

In [ ]:
def generate_extended_psd_signatures(n_frequencies: int = 14000) -> Tuple[np.ndarray, List[callable]]:
    """
    Generate extended library of PSD signature patterns.
    Now with 12 diverse signatures instead of 5.
    """
    frequencies = np.linspace(0, 6000, n_frequencies)  # 0 to 6 GHz
    
    def sig_1_narrow_peak_low(freqs):
        """Narrow peak at 800 MHz"""
        baseline = -80 * np.ones_like(freqs)
        peak = 30 * np.exp(-((freqs - 800)**2) / (40**2))
        return baseline + peak
    
    def sig_2_narrow_peak_mid(freqs):
        """Narrow peak at 2.4 GHz (WiFi-like)"""
        baseline = -80 * np.ones_like(freqs)
        peak = 35 * np.exp(-((freqs - 2400)**2) / (60**2))
        return baseline + peak
    
    def sig_3_narrow_peak_high(freqs):
        """Narrow peak at 5.2 GHz (5G WiFi-like)"""
        baseline = -80 * np.ones_like(freqs)
        peak = 32 * np.exp(-((freqs - 5200)**2) / (50**2))
        return baseline + peak
    
    def sig_4_dual_peaks(freqs):
        """Two peaks at 1.5 GHz and 3.5 GHz"""
        baseline = -80 * np.ones_like(freqs)
        peak1 = 28 * np.exp(-((freqs - 1500)**2) / (70**2))
        peak2 = 28 * np.exp(-((freqs - 3500)**2) / (70**2))
        return baseline + peak1 + peak2
    
    def sig_5_triple_peaks(freqs):
        """Three peaks (harmonics)"""
        baseline = -80 * np.ones_like(freqs)
        peak1 = 30 * np.exp(-((freqs - 1000)**2) / (50**2))
        peak2 = 25 * np.exp(-((freqs - 2000)**2) / (50**2))
        peak3 = 20 * np.exp(-((freqs - 3000)**2) / (50**2))
        return baseline + peak1 + peak2 + peak3
    
    def sig_6_broadband_low(freqs):
        """Broadband elevation 500-2000 MHz"""
        baseline = -80 * np.ones_like(freqs)
        elevated = 15 * (np.tanh((freqs - 500) / 100) - np.tanh((freqs - 2000) / 100))
        return baseline + elevated
    
    def sig_7_broadband_high(freqs):
        """Broadband elevation 3000-5000 MHz"""
        baseline = -80 * np.ones_like(freqs)
        elevated = 18 * (np.tanh((freqs - 3000) / 150) - np.tanh((freqs - 5000) / 150))
        return baseline + elevated
    
    def sig_8_swept_tone(freqs):
        """Linear chirp/swept frequency"""
        baseline = -80 * np.ones_like(freqs)
        # Simulate swept tone as series of peaks
        swept = np.zeros_like(freqs)
        for f_center in np.linspace(1000, 4000, 8):
            swept += 5 * np.exp(-((freqs - f_center)**2) / (100**2))
        return baseline + swept
    
    def sig_9_comb_spectrum(freqs):
        """Comb spectrum (regular spaced peaks)"""
        baseline = -80 * np.ones_like(freqs)
        comb = np.zeros_like(freqs)
        for f_center in np.arange(1000, 5000, 500):  # Every 500 MHz
            comb += 20 * np.exp(-((freqs - f_center)**2) / (30**2))
        return baseline + comb
    
    def sig_10_noise_like(freqs):
        """Elevated noise floor with bumps"""
        baseline = -75 * np.ones_like(freqs)  # Higher noise floor
        # Add some random-looking bumps (but deterministic)
        np.random.seed(123)
        for _ in range(20):
            f_center = np.random.uniform(500, 5500)
            baseline += np.random.uniform(2, 8) * np.exp(-((freqs - f_center)**2) / (200**2))
        np.random.seed(42)  # Reset
        return baseline
    
    def sig_11_slope_with_peak(freqs):
        """Sloped baseline with peak"""
        baseline = -85 + (freqs / 6000) * 15  # Upward slope
        peak = 30 * np.exp(-((freqs - 3500)**2) / (150**2))
        return baseline + peak
    
    def sig_12_interference_burst(freqs):
        """Multiple narrow interferers"""
        baseline = -80 * np.ones_like(freqs)
        interferers = np.zeros_like(freqs)
        for f_center in [900, 1800, 2100, 2600, 3500]:  # Cellular-like
            interferers += 25 * np.exp(-((freqs - f_center)**2) / (40**2))
        return baseline + interferers
    
    signatures = [
        sig_1_narrow_peak_low,
        sig_2_narrow_peak_mid,
        sig_3_narrow_peak_high,
        sig_4_dual_peaks,
        sig_5_triple_peaks,
        sig_6_broadband_low,
        sig_7_broadband_high,
        sig_8_swept_tone,
        sig_9_comb_spectrum,
        sig_10_noise_like,
        sig_11_slope_with_peak,
        sig_12_interference_burst
    ]
    
    return frequencies, signatures

## Multi-Sensor Data Generation

In [ ]:
class SensorCharacteristics:
    """
    Model different sensor characteristics that cause domain shift.
    """
    def __init__(self, 
                 sensor_id: int,
                 baseline_offset: float = 0.0,
                 gain_factor: float = 1.0,
                 noise_floor: float = 0.0,
                 frequency_shift: float = 0.0):
        self.sensor_id = sensor_id
        self.baseline_offset = baseline_offset  # dB offset
        self.gain_factor = gain_factor  # Multiplicative gain
        self.noise_floor = noise_floor  # Additive noise std
        self.frequency_shift = frequency_shift  # MHz shift
    
    def apply_to_psd(self, psd: np.ndarray, frequencies: np.ndarray) -> np.ndarray:
        """
        Apply sensor-specific characteristics to a PSD.
        """
        modified_psd = psd.copy()
        
        # Apply gain
        modified_psd = modified_psd * self.gain_factor
        
        # Apply baseline offset
        modified_psd = modified_psd + self.baseline_offset
        
        # Apply frequency shift (simplified - just shifts values)
        if self.frequency_shift != 0:
            shift_samples = int(self.frequency_shift / (frequencies[1] - frequencies[0]))
            modified_psd = np.roll(modified_psd, shift_samples)
        
        # Add sensor-specific noise
        if self.noise_floor > 0:
            modified_psd += np.random.normal(0, self.noise_floor, len(modified_psd))
        
        return modified_psd
    
    def __repr__(self):
        return (f"Sensor {self.sensor_id}: offset={self.baseline_offset:.1f}dB, "
                f"gain={self.gain_factor:.2f}, noise={self.noise_floor:.2f}dB, "
                f"freq_shift={self.frequency_shift:.0f}MHz")


def create_sensor_array(n_sensors: int, diversity_level: str = 'moderate') -> List[SensorCharacteristics]:
    """
    Create an array of sensors with varying characteristics.
    
    Args:
        n_sensors: Number of sensors to create
        diversity_level: 'low', 'moderate', or 'high' sensor variation
    """
    sensors = []
    
    # Define variation ranges based on diversity level
    if diversity_level == 'low':
        offset_range = (-2, 2)
        gain_range = (0.95, 1.05)
        noise_range = (0.5, 1.5)
        freq_shift_range = (-10, 10)
    elif diversity_level == 'moderate':
        offset_range = (-5, 5)
        gain_range = (0.9, 1.1)
        noise_range = (1.0, 2.5)
        freq_shift_range = (-20, 20)
    else:  # high
        offset_range = (-10, 10)
        gain_range = (0.8, 1.2)
        noise_range = (1.5, 4.0)
        freq_shift_range = (-50, 50)
    
    for i in range(n_sensors):
        sensor = SensorCharacteristics(
            sensor_id=i,
            baseline_offset=np.random.uniform(*offset_range),
            gain_factor=np.random.uniform(*gain_range),
            noise_floor=np.random.uniform(*noise_range),
            frequency_shift=np.random.uniform(*freq_shift_range)
        )
        sensors.append(sensor)
    
    return sensors


def generate_multi_sensor_dataset(
    sensors: List[SensorCharacteristics],
    n_samples_per_sensor: int,
    n_frequencies: int = 14000,
    base_noise_level: float = 0.0
) -> Dict:
    """
    Generate PSD data from multiple sensors.
    
    Returns:
        Dictionary with data for each sensor
    """
    frequencies, signature_funcs = generate_extended_psd_signatures(n_frequencies)
    n_signatures = len(signature_funcs)
    
    multi_sensor_data = {}
    
    for sensor in sensors:
        # Generate base PSDs (same distribution for all sensors)
        psd_data = np.zeros((n_samples_per_sensor, n_frequencies))
        labels = np.random.choice(n_signatures, size=n_samples_per_sensor)
        
        for i in range(n_samples_per_sensor):
            # Generate base signature
            sig_func = signature_funcs[labels[i]]
            base_psd = sig_func(frequencies)
            
            # Add base noise
            if base_noise_level > 0:
                base_psd += np.random.normal(0, base_noise_level, n_frequencies)
            
            # Apply sensor-specific characteristics
            psd_data[i] = sensor.apply_to_psd(base_psd, frequencies)
        
        multi_sensor_data[sensor.sensor_id] = {
            'psd_data': psd_data,
            'labels': labels,
            'sensor': sensor
        }
    
    return multi_sensor_data, frequencies, n_signatures

## Visualization Functions

In [ ]:
def plot_signature_library(frequencies: np.ndarray):
    """
    Visualize all signature types in the library.
    """
    _, signature_funcs = generate_extended_psd_signatures(len(frequencies))
    n_sigs = len(signature_funcs)
    
    n_cols = 3
    n_rows = (n_sigs + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
    axes = axes.flatten()
    
    for i, sig_func in enumerate(signature_funcs):
        psd = sig_func(frequencies)
        axes[i].plot(frequencies, psd, linewidth=2)
        axes[i].set_title(f'Signature {i}: {sig_func.__doc__.strip()}')
        axes[i].set_xlabel('Frequency (MHz)')
        axes[i].set_ylabel('Power (dBm)')
        axes[i].grid(True, alpha=0.3)
    
    # Hide unused subplots
    for i in range(n_sigs, len(axes)):
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()


def plot_sensor_comparison(multi_sensor_data: Dict, frequencies: np.ndarray, signature_idx: int = 0):
    """
    Plot same signature as measured by different sensors.
    """
    n_sensors = len(multi_sensor_data)
    
    fig, axes = plt.subplots(n_sensors, 1, figsize=(14, 3 * n_sensors))
    if n_sensors == 1:
        axes = [axes]
    
    for idx, (sensor_id, data) in enumerate(multi_sensor_data.items()):
        # Find samples with the target signature
        mask = data['labels'] == signature_idx
        if mask.sum() > 0:
            sample_psd = data['psd_data'][mask][0]
            axes[idx].plot(frequencies, sample_psd, linewidth=1.5)
            axes[idx].set_title(f"{data['sensor']}")
            axes[idx].set_xlabel('Frequency (MHz)')
            axes[idx].set_ylabel('Power (dBm)')
            axes[idx].grid(True, alpha=0.3)
    
    fig.suptitle(f'Same Signature (Type {signature_idx}) Across Different Sensors', fontsize=14, y=1.00)
    plt.tight_layout()
    plt.show()


def plot_cross_sensor_performance(results: Dict, metric: str = 'ari'):
    """
    Plot heatmap of cross-sensor performance.
    
    Args:
        results: Dictionary with performance metrics
        metric: 'ari' or 'nmi'
    """
    n_sensors = len(results)
    performance_matrix = np.zeros((n_sensors, n_sensors))
    
    for train_sensor in range(n_sensors):
        for test_sensor in range(n_sensors):
            key = f'train_{train_sensor}_test_{test_sensor}'
            if key in results:
                performance_matrix[train_sensor, test_sensor] = results[key].get(metric, 0)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(performance_matrix, annot=True, fmt='.3f', cmap='RdYlGn', 
                vmin=0, vmax=1, square=True, cbar_kws={'label': metric.upper()})
    plt.xlabel('Test Sensor')
    plt.ylabel('Train Sensor')
    plt.title(f'Cross-Sensor Performance ({metric.upper()})\nDiagonal = Same Sensor, Off-diagonal = Different Sensor')
    plt.tight_layout()
    plt.show()

## PCA and VAE Helper Functions

In [ ]:
def apply_pca_pipeline(train_data: np.ndarray, 
                       test_data: np.ndarray,
                       n_components: int = 10) -> Tuple:
    """
    Fit PCA on train data, apply to test data.
    """
    # Fit scaler and PCA on training data
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_data)
    
    pca = PCA(n_components=n_components)
    train_transformed = pca.fit_transform(train_scaled)
    
    # Apply to test data
    test_scaled = scaler.transform(test_data)
    test_transformed = pca.transform(test_scaled)
    
    return train_transformed, test_transformed, pca, scaler


class SimpleVAE(nn.Module):
    """Simplified VAE for faster training."""
    def __init__(self, input_dim: int = 14000, latent_dim: int = 10):
        super(SimpleVAE, self).__init__()
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.ReLU()
        )
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 512),
            nn.ReLU(),
            nn.Linear(512, input_dim)
        )
    
    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar


def train_vae_quick(vae, train_data, n_epochs=20, batch_size=64, lr=1e-3, verbose=False):
    """Quick VAE training for experiments."""
    dataset = TensorDataset(torch.FloatTensor(train_data))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    optimizer = optim.Adam(vae.parameters(), lr=lr)
    
    vae.train()
    for epoch in range(n_epochs):
        for batch in loader:
            data = batch[0].to(device)
            optimizer.zero_grad()
            
            recon, mu, logvar = vae(data)
            recon_loss = F.mse_loss(recon, data, reduction='sum')
            kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
            loss = recon_loss + kl_loss
            
            loss.backward()
            optimizer.step()
        
        if verbose and (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{n_epochs}, Loss: {loss.item()/len(data):.4f}")
    
    return vae


def extract_vae_features(vae, data, batch_size=128):
    """Extract latent features from VAE."""
    vae.eval()
    dataset = TensorDataset(torch.FloatTensor(data))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    features = []
    with torch.no_grad():
        for batch in loader:
            data = batch[0].to(device)
            mu, _ = vae.encode(data)
            features.append(mu.cpu().numpy())
    
    return np.vstack(features)


def evaluate_clustering(latent_features: np.ndarray, 
                       true_labels: np.ndarray,
                       eps: float = 2.0,
                       min_samples: int = 5) -> Dict:
    """
    Apply DBSCAN and evaluate clustering quality.
    """
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    cluster_labels = dbscan.fit_predict(latent_features)
    
    # Filter out noise points
    non_noise_mask = cluster_labels >= 0
    
    if non_noise_mask.sum() == 0:
        return {'ari': 0.0, 'nmi': 0.0, 'n_clusters': 0, 'n_noise': len(cluster_labels)}
    
    ari = adjusted_rand_score(true_labels[non_noise_mask], cluster_labels[non_noise_mask])
    nmi = normalized_mutual_info_score(true_labels[non_noise_mask], cluster_labels[non_noise_mask])
    
    n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    n_noise = list(cluster_labels).count(-1)
    
    return {
        'ari': ari,
        'nmi': nmi,
        'n_clusters': n_clusters,
        'n_noise': n_noise,
        'cluster_labels': cluster_labels
    }

---
# Experiment 1: Visualize Extended Signature Library

In [ ]:
# Generate and visualize all 12 signatures
N_FREQUENCIES = 14000
frequencies = np.linspace(0, 6000, N_FREQUENCIES)

print("Extended Signature Library: 12 Diverse PSD Patterns")
plot_signature_library(frequencies)

---
# Experiment 2: Multi-Sensor Data Generation

In [ ]:
# Create sensor array with moderate diversity
N_SENSORS = 4
N_SAMPLES_PER_SENSOR = 500
BASE_NOISE = 1.0

sensors = create_sensor_array(N_SENSORS, diversity_level='moderate')

print("Sensor Array Characteristics:")
print("=" * 80)
for sensor in sensors:
    print(sensor)
print("=" * 80)

In [ ]:
# Generate multi-sensor dataset
print(f"\nGenerating data for {N_SENSORS} sensors...")
multi_sensor_data, frequencies, n_signatures = generate_multi_sensor_dataset(
    sensors=sensors,
    n_samples_per_sensor=N_SAMPLES_PER_SENSOR,
    n_frequencies=N_FREQUENCIES,
    base_noise_level=BASE_NOISE
)

print(f"Generated {N_SAMPLES_PER_SENSOR} samples per sensor")
print(f"Total signatures in library: {n_signatures}")
print(f"PSD dimension: {N_FREQUENCIES}")

## Visualize Same Signature Across Different Sensors

In [ ]:
# Show how the same signature looks different on different sensors
print("Notice how the same signature appears different on each sensor due to:")
print("- Baseline offset (vertical shift)")
print("- Gain differences (amplitude scaling)")
print("- Noise characteristics")
print("- Frequency calibration shifts")
print()

plot_sensor_comparison(multi_sensor_data, frequencies, signature_idx=1)

---
# Experiment 3: Cross-Sensor PCA Performance

**Test**: Train PCA on Sensor 0, apply to all sensors (including itself)

In [ ]:
# PCA hyperparameters
N_COMPONENTS = 10
DBSCAN_EPS = 2.0
DBSCAN_MIN_SAMPLES = 5

print("PCA CROSS-SENSOR EXPERIMENT")
print("=" * 80)
print("Training PCA on Sensor 0, testing on all sensors...")
print()

pca_results = {}

# Train on Sensor 0
train_sensor_id = 0
train_data = multi_sensor_data[train_sensor_id]['psd_data']

# Fit PCA on Sensor 0
scaler_pca = StandardScaler()
train_scaled = scaler_pca.fit_transform(train_data)
pca = PCA(n_components=N_COMPONENTS)
pca.fit(train_scaled)

print(f"PCA trained on Sensor {train_sensor_id}")
print(f"Explained variance: {pca.explained_variance_ratio_.sum():.4f}")
print()

# Test on all sensors
for test_sensor_id in range(N_SENSORS):
    test_data = multi_sensor_data[test_sensor_id]['psd_data']
    test_labels = multi_sensor_data[test_sensor_id]['labels']
    
    # Transform test data with Sensor 0's PCA
    test_scaled = scaler_pca.transform(test_data)
    test_transformed = pca.transform(test_scaled)
    
    # Cluster and evaluate
    eval_results = evaluate_clustering(test_transformed, test_labels, 
                                       eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES)
    
    pca_results[f'train_{train_sensor_id}_test_{test_sensor_id}'] = eval_results
    
    status = "✓ SAME SENSOR" if test_sensor_id == train_sensor_id else "✗ DIFFERENT SENSOR"
    print(f"Test on Sensor {test_sensor_id} {status}:")
    print(f"  ARI: {eval_results['ari']:.4f}, NMI: {eval_results['nmi']:.4f}, "
          f"Clusters: {eval_results['n_clusters']}, Noise: {eval_results['n_noise']}")

print("\n" + "=" * 80)
print("KEY OBSERVATION:")
print("Performance degrades significantly when applying PCA trained on one sensor to another!")
print("=" * 80)

---
# Experiment 4: Cross-Sensor VAE Performance

**Test**: Train VAE on Sensor 0, apply to all sensors

In [ ]:
# VAE hyperparameters
LATENT_DIM = 10
VAE_EPOCHS = 20
BATCH_SIZE = 64

print("VAE CROSS-SENSOR EXPERIMENT")
print("=" * 80)
print("Training VAE on Sensor 0, testing on all sensors...")
print()

vae_results = {}

# Train VAE on Sensor 0
train_sensor_id = 0
train_data = multi_sensor_data[train_sensor_id]['psd_data']

scaler_vae = StandardScaler()
train_scaled = scaler_vae.fit_transform(train_data)

vae = SimpleVAE(input_dim=N_FREQUENCIES, latent_dim=LATENT_DIM).to(device)
vae = train_vae_quick(vae, train_scaled, n_epochs=VAE_EPOCHS, batch_size=BATCH_SIZE, verbose=True)

print(f"\nVAE trained on Sensor {train_sensor_id}")
print()

# Test on all sensors
for test_sensor_id in range(N_SENSORS):
    test_data = multi_sensor_data[test_sensor_id]['psd_data']
    test_labels = multi_sensor_data[test_sensor_id]['labels']
    
    # Transform test data with Sensor 0's VAE
    test_scaled = scaler_vae.transform(test_data)
    test_features = extract_vae_features(vae, test_scaled, batch_size=BATCH_SIZE)
    
    # Cluster and evaluate
    eval_results = evaluate_clustering(test_features, test_labels,
                                       eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES)
    
    vae_results[f'train_{train_sensor_id}_test_{test_sensor_id}'] = eval_results
    
    status = "✓ SAME SENSOR" if test_sensor_id == train_sensor_id else "✗ DIFFERENT SENSOR"
    print(f"Test on Sensor {test_sensor_id} {status}:")
    print(f"  ARI: {eval_results['ari']:.4f}, NMI: {eval_results['nmi']:.4f}, "
          f"Clusters: {eval_results['n_clusters']}, Noise: {eval_results['n_noise']}")

print("\n" + "=" * 80)
print("KEY OBSERVATION:")
print("VAE also struggles with cross-sensor transfer!")
print("=" * 80)

---
# Experiment 5: Solution 1 - Per-Sensor Models

**Strategy**: Train separate PCA/VAE models for each sensor

In [ ]:
print("SOLUTION 1: PER-SENSOR PCA MODELS")
print("=" * 80)
print("Training separate PCA for each sensor...")
print()

per_sensor_pca_results = {}

for sensor_id in range(N_SENSORS):
    data = multi_sensor_data[sensor_id]['psd_data']
    labels = multi_sensor_data[sensor_id]['labels']
    
    # Fit PCA on this sensor's data
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data)
    pca = PCA(n_components=N_COMPONENTS)
    data_transformed = pca.fit_transform(data_scaled)
    
    # Evaluate
    eval_results = evaluate_clustering(data_transformed, labels,
                                       eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES)
    
    per_sensor_pca_results[f'sensor_{sensor_id}'] = eval_results
    
    print(f"Sensor {sensor_id}: ARI={eval_results['ari']:.4f}, NMI={eval_results['nmi']:.4f}")

print("\n" + "=" * 80)
print("RESULT: Per-sensor models work well!")
print("BUT: Requires retraining for each new sensor (not scalable to 100s of sensors)")
print("=" * 80)

---
# Experiment 6: Solution 2 - Pooled Multi-Sensor Training

**Strategy**: Train single model on combined data from all sensors

In [ ]:
print("SOLUTION 2: POOLED MULTI-SENSOR PCA")
print("=" * 80)
print("Training PCA on combined data from all sensors...")
print()

# Pool all sensor data
pooled_data = []
pooled_labels = []

for sensor_id in range(N_SENSORS):
    pooled_data.append(multi_sensor_data[sensor_id]['psd_data'])
    pooled_labels.append(multi_sensor_data[sensor_id]['labels'])

pooled_data = np.vstack(pooled_data)
pooled_labels = np.hstack(pooled_labels)

print(f"Pooled dataset: {pooled_data.shape[0]} samples from {N_SENSORS} sensors")

# Train PCA on pooled data
scaler_pooled = StandardScaler()
pooled_scaled = scaler_pooled.fit_transform(pooled_data)
pca_pooled = PCA(n_components=N_COMPONENTS)
pooled_transformed = pca_pooled.fit_transform(pooled_scaled)

print(f"Pooled PCA explained variance: {pca_pooled.explained_variance_ratio_.sum():.4f}")
print()

# Evaluate on pooled data
pooled_eval = evaluate_clustering(pooled_transformed, pooled_labels,
                                  eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES)

print(f"Pooled clustering: ARI={pooled_eval['ari']:.4f}, NMI={pooled_eval['nmi']:.4f}")
print()

# Test on individual sensors
print("Testing pooled model on individual sensors:")
for sensor_id in range(N_SENSORS):
    data = multi_sensor_data[sensor_id]['psd_data']
    labels = multi_sensor_data[sensor_id]['labels']
    
    data_scaled = scaler_pooled.transform(data)
    data_transformed = pca_pooled.transform(data_scaled)
    
    eval_results = evaluate_clustering(data_transformed, labels,
                                       eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES)
    
    print(f"  Sensor {sensor_id}: ARI={eval_results['ari']:.4f}, NMI={eval_results['nmi']:.4f}")

print("\n" + "=" * 80)
print("RESULT: Pooled training provides moderate performance across all sensors")
print("ADVANTAGE: Single model works for new sensors (if they have similar characteristics)")
print("LIMITATION: May not capture sensor-specific nuances as well")
print("=" * 80)

---
# Experiment 7: Solution 3 - Sensor-Agnostic Normalization

**Strategy**: Normalize each PSD to remove sensor-specific characteristics before modeling

In [ ]:
def normalize_psd(psd: np.ndarray) -> np.ndarray:
    """
    Normalize PSD to be sensor-agnostic:
    1. Subtract baseline (median)
    2. Divide by scale (MAD - median absolute deviation)
    """
    baseline = np.median(psd)
    mad = np.median(np.abs(psd - baseline))
    if mad == 0:
        mad = 1.0
    return (psd - baseline) / mad


print("SOLUTION 3: SENSOR-AGNOSTIC NORMALIZATION")
print("=" * 80)
print("Normalizing each PSD to remove sensor-specific offsets and gains...")
print()

# Normalize all sensor data
normalized_multi_sensor = {}
for sensor_id in range(N_SENSORS):
    data = multi_sensor_data[sensor_id]['psd_data']
    normalized_data = np.array([normalize_psd(psd) for psd in data])
    normalized_multi_sensor[sensor_id] = {
        'psd_data': normalized_data,
        'labels': multi_sensor_data[sensor_id]['labels']
    }

# Train PCA on normalized Sensor 0 data
train_sensor_id = 0
train_data_norm = normalized_multi_sensor[train_sensor_id]['psd_data']

scaler_norm = StandardScaler()
train_scaled_norm = scaler_norm.fit_transform(train_data_norm)
pca_norm = PCA(n_components=N_COMPONENTS)
pca_norm.fit(train_scaled_norm)

print(f"PCA trained on normalized Sensor {train_sensor_id} data")
print()

# Test on all sensors
print("Testing on normalized data from all sensors:")
norm_results = {}

for test_sensor_id in range(N_SENSORS):
    test_data_norm = normalized_multi_sensor[test_sensor_id]['psd_data']
    test_labels = normalized_multi_sensor[test_sensor_id]['labels']
    
    test_scaled_norm = scaler_norm.transform(test_data_norm)
    test_transformed_norm = pca_norm.transform(test_scaled_norm)
    
    eval_results = evaluate_clustering(test_transformed_norm, test_labels,
                                       eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES)
    
    norm_results[f'test_{test_sensor_id}'] = eval_results
    
    status = "✓ SAME SENSOR" if test_sensor_id == train_sensor_id else "✗ DIFFERENT SENSOR"
    print(f"  Sensor {test_sensor_id} {status}: ARI={eval_results['ari']:.4f}, NMI={eval_results['nmi']:.4f}")

print("\n" + "=" * 80)
print("RESULT: Normalization significantly improves cross-sensor transferability!")
print("ADVANTAGE: Simple preprocessing step, no retraining needed")
print("LIMITATION: May not handle all sensor differences (e.g., frequency shifts)")
print("=" * 80)

---
# Summary: Multi-Sensor Deployment Strategies

## Problem Recap

**Question**: Can we train on Sensor 1 and apply to Sensors 2-10 without retraining?

**Answer**: **Not directly** - both PCA and VAE struggle with domain shift between sensors.

## Solutions Comparison

### Strategy 1: Per-Sensor Models
- ✅ **Best performance** on each sensor
- ✅ Accounts for all sensor-specific characteristics
- ❌ Requires labeled data from each sensor
- ❌ Not scalable (need to retrain for 100s of sensors)
- ❌ Can't handle new sensors immediately

**When to use**: Few sensors (<10), have labeled data per sensor, need maximum accuracy

### Strategy 2: Pooled Multi-Sensor Training
- ✅ Single model for all sensors
- ✅ Generalizes to new sensors with similar characteristics
- ✅ Scalable to many sensors
- ⚠️ Moderate performance (not optimal for any single sensor)
- ❌ Requires representative data from multiple sensors

**When to use**: Many sensors, expect new sensors, willing to sacrifice some accuracy for generalization

### Strategy 3: Sensor-Agnostic Normalization + Single Model
- ✅ **Best of both worlds** - good performance + transferability
- ✅ Simple preprocessing step
- ✅ No retraining needed for new sensors
- ✅ Works well if sensors differ mainly in baseline/gain
- ⚠️ May not handle complex differences (frequency shift, nonlinear distortion)

**When to use**: Many sensors with similar characteristics, need quick deployment, limited labeled data

### Strategy 4: Hybrid Approach (Recommended)
1. **Normalize** all PSDs to remove offset/gain differences
2. **Train on pooled data** from representative sensors
3. **Fine-tune** on new sensors if performance is insufficient

## Practical Recommendations

1. **Start with normalization** - Always apply sensor-agnostic normalization
2. **Pilot with pooled training** - Train on 3-5 representative sensors
3. **Monitor performance** - Track clustering quality on new sensors
4. **Adapt as needed** - Retrain if new sensors show significantly different characteristics

## Hyperparameter Transferability

✅ **Usually transferable**:
- Number of PCA components / latent dimensions
- DBSCAN min_samples
- VAE architecture (if using normalization)

⚠️ **May need adjustment**:
- DBSCAN eps (depends on sensor noise level)
- VAE beta (KL weight)

❌ **Never transferable**:
- The fitted PCA components themselves
- The trained VAE weights
- StandardScaler parameters (unless using pooled training)